# Replication: Imai and Ratkovic (2015)

**Robust Estimation of Inverse Probability Weights for Marginal Structural Models**  
*Journal of the American Statistical Association*, 110(511), 1013–1023.

This notebook reproduces the empirical application from Section 5 of the paper, which analyzes the effect of negative campaign advertising on Democratic vote share using the Blackwell (2013) longitudinal dataset.

- **Section 5** — Negative campaign advertising and election outcomes (Table 3)
- **Section 5** — Covariate balance diagnostics (Figure 4 concept)

The dataset contains 114 U.S. Senate and gubernatorial races from 2000–2006, observed over $J = 5$ weekly periods leading up to each election. The treatment is whether a candidate ran negative advertisements in a given week, and the outcome is the Democratic candidate's final vote share.

In [1]:
import warnings
import numpy as np
import pandas as pd
import statsmodels.api as sm

import cbps
from cbps.datasets import load_blackwell

## Data

The Blackwell (2013) panel dataset tracks negative campaign advertising across weekly periods before U.S. elections. The treatment model includes lagged treatment indicators, campaign characteristics, and year fixed effects.

In [2]:
FULL_FORMULA = (
    'd.gone.neg ~ d.gone.neg.l1 + d.gone.neg.l2 + d.neg.frac.l3'
    ' + camp.length + deminc + base.poll'
    ' + year.2002 + year.2004 + year.2006'
    ' + base.und + office'
)

df = load_blackwell()
n_units = df['demName'].nunique()
periods = sorted(df['time'].unique())

print(f'Blackwell (2013) negative campaign advertising data')
print(f'  Candidates: {n_units}, periods: {len(periods)}, observations: {len(df)}')
for t in periods:
    prev = df.loc[df['time'] == t, 'd.gone.neg'].mean()
    print(f'  Week {t} treatment prevalence: {prev:.3f}')

Blackwell (2013) negative campaign advertising data
  Candidates: 114, periods: 5, observations: 570
  Week 1 treatment prevalence: 0.667
  Week 2 treatment prevalence: 0.737
  Week 3 treatment prevalence: 0.789
  Week 4 treatment prevalence: 0.833
  Week 5 treatment prevalence: 0.395


## Weight Estimation

Marginal structural model weights are estimated via two variants of CBMSM:

- **CBPS-Approx**: low-rank (approximate) variance, computationally efficient
- **CBPS**: full variance matrix

GLM (standard logistic regression) weights are extracted from the `glm_weights` attribute of the CBMSM fit for comparison.

In [3]:
fits = {}

print('Estimating CBPS-Approximate (low-rank variance)...')
fits['CBPS-Approx'] = cbps.CBMSM(
    formula=FULL_FORMULA, id='demName', time='time', data=df,
    type='MSM', time_vary=True, twostep=True, msm_variance='approx',
)

print('Estimating CBPS (full variance)...')
with warnings.catch_warnings():
    warnings.simplefilter('ignore')
    fits['CBPS'] = cbps.CBMSM(
        formula=FULL_FORMULA, id='demName', time='time', data=df,
        type='MSM', time_vary=True, twostep=True, msm_variance='full',
    )

for name, fit in fits.items():
    w = fit.fitted_values
    print(f'  {name}: converged={fit.converged}, J={fit.J:.4f}, '
          f'weights=[{w.min():.4f}, {w.max():.4f}]')

Estimating CBPS-Approximate (low-rank variance)...


/Users/cxy/Desktop/cbps/CBPS_python/cbps/core/cbps_binary.py:1547: UserWarning: method='exact': Moment conditions converged to 9.34e-04, below theoretical GMM precision <1e-10. This is a known limitation of balance loss optimization.
For exact moment=0 satisfaction (~1e-15 precision), use theoretical_exact=True in CBPS() call.
  warnings.warn(
/Users/cxy/Desktop/cbps/CBPS_python/cbps/msm/cbmsm.py:1563: UserWarning: 5 observations (4.4%) at probability boundaries. This is usually acceptable if <10%, but check covariate balance.
  cbps_sub = cbps_binary_fit(
/Users/cxy/Desktop/cbps/CBPS_python/cbps/core/cbps_binary.py:1547: UserWarning: method='exact': Moment conditions converged to 3.34e-02, below theoretical GMM precision <1e-10. This is a known limitation of balance loss optimization.
For exact moment=0 satisfaction (~1e-15 precision), use theoretical_exact=True in CBPS() call.
  warnings.warn(
/Users/cxy/Desktop/cbps/CBPS_python/cbps/msm/cbmsm.py:1563: UserWarning: 1 observations (0.

Estimating CBPS (full variance)...
  CBPS-Approx: converged=False, J=27.2289, weights=[0.0001, 0.1853]
  CBPS: converged=True, J=0.5255, weights=[0.0001, 0.2167]


## Model Summary

All result classes now provide a consistent `summary()` method that returns a dedicated summary object (not a raw string). This enables programmatic access to summary data while preserving the familiar `print()` output.

In [ ]:
# CBMSMSummary object — type is CBMSMSummary, not str
summary_obj = fits['CBPS-Approx'].summary()
print(f'Summary type: {type(summary_obj).__name__}')
print()
print(summary_obj)

## Weight Diagnostics

In [4]:
def get_weights(fits, method):
    if method == 'GLM':
        return fits['CBPS-Approx'].glm_weights
    return fits[method].fitted_values


methods = ['GLM', 'CBPS', 'CBPS-Approx']
ws = {m: get_weights(fits, m) for m in methods}

stats = [
    ('Min', np.min), ('Q1', lambda x: np.percentile(x, 25)),
    ('Median', np.median), ('Mean', np.mean),
    ('Q3', lambda x: np.percentile(x, 75)),
    ('Max', np.max), ('Std Dev', np.std),
]

header = f"{'Statistic':<12}" + ''.join(f'  {m:>14}' for m in methods)
print(header)
print('-' * len(header))
for label, fn in stats:
    row = f'{label:<12}' + ''.join(f'  {fn(ws[m]):>14.4f}' for m in methods)
    print(row)

Statistic                GLM            CBPS     CBPS-Approx
------------------------------------------------------------
Min                   0.0000          0.0001          0.0001
Q1                    0.0099          0.0097          0.0104
Median                0.0966          0.0577          0.0516
Mean                  0.1179          0.0780          0.0636
Q3                    0.2234          0.1347          0.1113
Max                   0.2792          0.2167          0.1853
Std Dev               0.1024          0.0702          0.0577


## Table 3: Impact of Negative Advertising on Vote Share

Weighted least squares regression of Democratic vote share on treatment history indicators (time-specific effects $\beta_1$ through $\beta_5$) and cumulative treatment count. Standard errors in parentheses.

In [5]:
first_t = df['time'].min()
outcome = df.loc[df['time'] == first_t, 'demprcnt'].values

table = {}
for method in methods:
    fit = fits['CBPS-Approx'] if method == 'GLM' else fits[method]
    w = get_weights(fits, method)

    # Time-specific effects
    X_hist = sm.add_constant(fit.treat_hist)
    m_hist = sm.WLS(outcome, X_hist, weights=w).fit()

    effects = {}
    for j in range(fit.treat_hist.shape[1]):
        effects[f'beta_{j+1}'] = (m_hist.params[j+1], m_hist.bse[j+1])

    # Cumulative effect
    X_cum = sm.add_constant(fit.treat_cum.reshape(-1, 1))
    m_cum = sm.WLS(outcome, X_cum, weights=w).fit()
    effects['Cumulative'] = (m_cum.params[1], m_cum.bse[1])

    table[method] = effects

# Display
print('Table 3: Impact of Negative Advertising on Vote Share')
print('  (Imai and Ratkovic, 2015, Section 5)\n')

header = f"{'Effect':<14}" + ''.join(f'  {m:>18}' for m in methods)
print(header)
print('-' * len(header))
for key in [f'beta_{j}' for j in range(1, 6)] + ['Cumulative']:
    row = f'{key:<14}'
    for method in methods:
        coef, se = table[method][key]
        row += f'  {coef:>8.3f} ({se:.3f})'
    print(row)

print()
print('Notes: WLS of Democratic vote share on treatment history indicators')
print('(time-specific) or cumulative treatment count. SE in parentheses.')

Table 3: Impact of Negative Advertising on Vote Share
  (Imai and Ratkovic, 2015, Section 5)

Effect                         GLM                CBPS         CBPS-Approx
--------------------------------------------------------------------------
beta_1             0.714 (5.607)     2.874 (5.125)     2.376 (4.293)
beta_2             6.719 (11.290)     5.185 (9.723)     5.319 (9.653)
beta_3            -3.609 (16.081)    -5.136 (12.936)    -3.058 (13.237)
beta_4           -10.769 (12.759)    -8.718 (10.094)   -10.068 (10.233)
beta_5            -2.057 (0.934)    -2.224 (0.962)    -2.027 (1.007)
Cumulative        -1.767 (0.412)    -1.569 (0.404)    -1.438 (0.434)

Notes: WLS of Democratic vote share on treatment history indicators
(time-specific) or cumulative treatment count. SE in parentheses.


## Covariate Balance Assessment (cf. Figure 4)

Per-covariate mean absolute standardized differences under GLM and CBPS weighting, and the fraction of balancing conditions where CBPS achieves better balance.

In [6]:
fit = fits['CBPS-Approx']
bal = fit.balance()

glm_raw = bal['Unweighted']
cbps_raw = bal['Balanced']
cov_names = bal['row_names']

n_pat = glm_raw.shape[1] // 2
glm_std = np.abs(glm_raw[:, n_pat:])
cbps_std = np.abs(cbps_raw[:, n_pat:])

glm_avg = glm_std.mean(axis=1)
cbps_avg = cbps_std.mean(axis=1)

print(f"{'Covariate':<20} {'GLM':>10} {'CBPS':>10} {'Diff':>10}")
print('-' * 52)
for i, name in enumerate(cov_names):
    d = glm_avg[i] - cbps_avg[i]
    print(f"{name:<20} {glm_avg[i]:>10.4f} {cbps_avg[i]:>10.4f} "
          f"{'+'if d > 0 else ''}{d:>9.4f}")

n_cond = glm_std.size
n_better = int(np.sum(cbps_std < glm_std))
pct = 100.0 * n_better / n_cond if n_cond > 0 else 0.0

print()
print(f'Balancing conditions: {n_cond}')
print(f'CBPS improves balance: {n_better}/{n_cond} ({pct:.1f}%)')
print(f'Mean abs. imbalance -- GLM: {glm_std.mean():.4f}, CBPS: {cbps_std.mean():.4f}')

Covariate                   GLM       CBPS       Diff
----------------------------------------------------
X1                       4.6639     0.0527 +   4.6112
X2                       2.5498     0.0477 +   2.5021
X3                       8.2167     0.0919 +   8.1247
X4                       1.7199     0.0308 +   1.6891
X5                       8.7741     0.1031 +   8.6710
X6                       6.3295     0.0556 +   6.2739
X7                       1.8821     0.0366 +   1.8455
X8                       2.3795     0.0401 +   2.3394
X9                       9.9155     0.0786 +   9.8369
X10                      4.8060     0.0662 +   4.7398

Balancing conditions: 180
CBPS improves balance: 121/180 (67.2%)
Mean abs. imbalance -- GLM: 5.1237, CBPS: 0.0603
